# Home Credit Proxy Notebook 06 — Product Pack & Policy Simulator

This notebook turns the stage models and registry artifacts into a **product-facing inference pack** and a **policy simulator**.

## Goals
1. Load notebook 4 and notebook 5 artifacts
2. Build a unified stage-aware inference flow
3. Simulate product policy actions for stages A / B / C
4. Export handoff assets for Streamlit / API / business review

## Business framing
- **Stage A** = starter-offer / onboarding recommendation
- **Stage B** = main decision champion
- **Stage C** = collection / monitoring guardrail

> Higher `score_300_900` means **higher risk**, not higher credit quality.


**Notebook 6 revised goals:** reduce Stage A review overload and make inference fallback explicit instead of returning NaN scores.

In [1]:

# =========================================================
# 0. Imports
# =========================================================
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

TARGET_COL = "target"
SCORE_COL = "score_300_900"
PROBA_COL = "score_proba"
STAGE_COL = "stage"


In [2]:

# =========================================================
# 1. Locate notebook 5 and notebook 4 artifact directories
# =========================================================
DEFAULT_INPUT_DIR_NB5 = Path("/kaggle/working/homecredit_proxy_notebook_05_benchmark_registry")

required_markers_nb5 = [
    "stage_benchmark_summary.csv",
    "stage_policy_recommendation.csv",
    "model_registry_manifest.json",
    "model_registry_pack.joblib",
]

candidate_roots_nb5 = [
    DEFAULT_INPUT_DIR_NB5,
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    Path("/mnt/data"),
    Path.cwd(),
]

INPUT_DIR_NB5 = None
for root in candidate_roots_nb5:
    if root.is_dir() and all((root / f).exists() for f in required_markers_nb5):
        INPUT_DIR_NB5 = root
        break

if INPUT_DIR_NB5 is None:
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data"), Path.cwd()]:
        if not root.exists():
            continue
        for p in root.rglob("model_registry_manifest.json"):
            parent = p.parent
            if all((parent / f).exists() for f in required_markers_nb5):
                INPUT_DIR_NB5 = parent
                break
        if INPUT_DIR_NB5 is not None:
            break

if INPUT_DIR_NB5 is None:
    raise FileNotFoundError(
        "Cannot locate notebook 5 artifact folder. "
        "Expected files: stage_benchmark_summary.csv, stage_policy_recommendation.csv, "
        "model_registry_manifest.json, model_registry_pack.joblib"
    )

with open(INPUT_DIR_NB5 / "model_registry_manifest.json", "r", encoding="utf-8") as f:
    registry_manifest = json.load(f)

source_nb4_dir = registry_manifest.get("source_notebook_4_dir")
candidate_nb4 = [Path(source_nb4_dir)] if source_nb4_dir else []
candidate_nb4 += [
    Path("/kaggle/working/homecredit_proxy_notebook_04_business"),
    Path("/kaggle/input"),
    Path("/mnt/data"),
    Path.cwd(),
]

required_markers_nb4 = [
    "stage_model_metrics.csv",
    "stage_business_readiness.csv",
    "product_scoring_manifest.json",
    "unified_stage_inference_pack.joblib",
]

INPUT_DIR_NB4 = None
for root in candidate_nb4:
    if root.is_dir() and all((root / f).exists() for f in required_markers_nb4):
        INPUT_DIR_NB4 = root
        break

if INPUT_DIR_NB4 is None:
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("/mnt/data"), Path.cwd()]:
        if not root.exists():
            continue
        for p in root.rglob("unified_stage_inference_pack.joblib"):
            parent = p.parent
            if all((parent / f).exists() for f in required_markers_nb4):
                INPUT_DIR_NB4 = parent
                break
        if INPUT_DIR_NB4 is not None:
            break

if INPUT_DIR_NB4 is None:
    raise FileNotFoundError(
        "Cannot locate notebook 4 artifact folder. "
        "Expected files: stage_model_metrics.csv, stage_business_readiness.csv, "
        "product_scoring_manifest.json, unified_stage_inference_pack.joblib"
    )

WORK_DIR = Path("/kaggle/working/homecredit_proxy_notebook_06_product_pack")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("INPUT_DIR_NB5:", INPUT_DIR_NB5)
print("INPUT_DIR_NB4:", INPUT_DIR_NB4)
print("WORK_DIR     :", WORK_DIR)


INPUT_DIR_NB5: /kaggle/input/notebooks/hongvittrnh/notebook-5/homecredit_proxy_notebook_05_benchmark_registry
INPUT_DIR_NB4: /kaggle/input/notebooks/hongvittrnh/notebook-4-0804/homecredit_proxy_notebook_04_business
WORK_DIR     : /kaggle/working/homecredit_proxy_notebook_06_product_pack


In [3]:

# =========================================================
# 2. Load notebook 4 / notebook 5 artifacts
# =========================================================
benchmark_df = pl.read_csv(INPUT_DIR_NB5 / "stage_benchmark_summary.csv")
policy_recommendation_df = pl.read_csv(INPUT_DIR_NB5 / "stage_policy_recommendation.csv")
registry_df = pl.read_csv(INPUT_DIR_NB5 / "model_registry_table.csv")
metrics_df = pl.read_csv(INPUT_DIR_NB4 / "stage_model_metrics.csv")
readiness_df = pl.read_csv(INPUT_DIR_NB4 / "stage_business_readiness.csv")

with open(INPUT_DIR_NB5 / "model_registry_manifest.json", "r", encoding="utf-8") as f:
    model_registry_manifest = json.load(f)

with open(INPUT_DIR_NB4 / "product_scoring_manifest.json", "r", encoding="utf-8") as f:
    product_scoring_manifest = json.load(f)

registry_pack = joblib.load(INPUT_DIR_NB5 / "model_registry_pack.joblib")
inference_pack = joblib.load(INPUT_DIR_NB4 / "unified_stage_inference_pack.joblib")

stage_valid_scored = {}
stage_product_scored = {}
stage_zone_summary = {}

for stage in ["A", "B", "C"]:
    valid_path = INPUT_DIR_NB4 / f"stage_{stage.lower()}_valid_scored.parquet"
    product_path = INPUT_DIR_NB4 / f"stage_{stage.lower()}_product_scored.parquet"
    zone_path = INPUT_DIR_NB5 / f"stage_{stage.lower()}_decision_zone_summary.csv"

    if valid_path.exists():
        stage_valid_scored[stage] = pl.read_parquet(valid_path)
    if product_path.exists():
        stage_product_scored[stage] = pl.read_parquet(product_path)
    if zone_path.exists():
        stage_zone_summary[stage] = pl.read_csv(zone_path)

print("Loaded valid scored stages :", sorted(stage_valid_scored.keys()))
print("Loaded product scored stages:", sorted(stage_product_scored.keys()))
print("Loaded zone summaries      :", sorted(stage_zone_summary.keys()))

display(policy_recommendation_df)
display(registry_df)


Loaded valid scored stages : ['A', 'B', 'C']
Loaded product scored stages: ['A', 'B', 'C']
Loaded zone summaries      : ['A', 'B', 'C']


stage,policy_mode,review_threshold_score_300_900,reject_threshold_score_300_900,n_valid_rows,avg_score_valid,bad_rate_valid,recommended_role,note
str,str,f64,f64,i64,f64,f64,str,str
"""A""","""starter_offer_only""",814.777871,829.939654,63723,809.169269,0.021217,"""starter_offer""","""Stage A is thin-file / new-to-…"
"""B""","""full_decision""",742.761886,746.833473,217006,736.031725,0.021976,"""decision_champion""","""Stage B is the primary decisio…"
"""C""","""guardrail_only""",724.997459,796.199662,24602,723.491787,0.073205,"""collection_guardrail""","""Stage C should be used for gua…"


stage,registry_role,stage_name,policy_mode,ready_for_nb5_business_strict,ready_for_product_scoring_api,n_features_kept,auc_valid,ks_valid,pr_auc_valid,review_threshold_score_300_900,reject_threshold_score_300_900,recommended_role,recommended_use,score_note,artifact_model_path,artifact_valid_scored_path,artifact_product_scored_path,artifact_band_table_path
str,str,str,str,bool,bool,i64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str
"""A""","""active_guarded_model""","""Application / New-to-bank Star…","""starter_offer_only""",false,true,2,0.546369,0.122847,0.026586,814.777871,829.939654,"""starter_offer""","""starter loan recommendation + …","""Prefer score_raw / score_300_9…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…"
"""B""","""champion_decision_model""","""Behavior / Thin-file""","""full_decision""",true,true,6,0.627878,0.22536,0.034724,742.761886,746.833473,"""decision_champion""","""ranking + cutoff exploration +…","""Prefer score_raw / score_300_9…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…"
"""C""","""active_guarded_model""","""Collection / Mature risk""","""guardrail_only""",false,true,6,0.627189,0.23954,0.106594,724.997459,796.199662,"""collection_guardrail""","""high-risk prioritization + col…","""Prefer score_raw / score_300_9…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…"


In [4]:

# =========================================================
# 3. Helper functions
# =========================================================
def safe_div(num: float, den: float) -> float:
    return float(num / den) if den not in [0, 0.0, None] else float("nan")

def first_existing(d: dict, keys: list[str], default=None):
    for k in keys:
        if k in d and d[k] is not None and not (isinstance(d[k], float) and math.isnan(d[k])):
            return d[k]
    return default

def np_quantile_safe(arr: np.ndarray, q: float) -> float:
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return float("nan")
    return float(np.quantile(arr, q))

def sigmoid(x: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / (1.0 + np.exp(-x))

def clamp_proba(x: float) -> float:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return float("nan")
    return float(min(1.0, max(0.0, x)))

def proba_to_score_300_900(proba: float) -> float:
    if proba is None or (isinstance(proba, float) and math.isnan(proba)):
        return float("nan")
    return float(300.0 + 600.0 * proba)

def score_to_proba_300_900(score: float) -> float:
    if score is None or (isinstance(score, float) and math.isnan(score)):
        return float("nan")
    return clamp_proba((float(score) - 300.0) / 600.0)

def stage_config(stage: str) -> dict:
    rows = policy_recommendation_df.filter(pl.col("stage") == stage).to_dicts()
    return rows[0] if rows else {}

def stage_score_reference(stage: str) -> dict:
    df = stage_valid_scored.get(stage)
    if df is None or SCORE_COL not in df.columns:
        return {}
    use = df.select([c for c in [SCORE_COL, TARGET_COL] if c in df.columns]).drop_nulls()
    if use.height == 0:
        return {}
    scores = use[SCORE_COL].to_numpy()
    out = {
        "q15": np_quantile_safe(scores, 0.15),
        "q30": np_quantile_safe(scores, 0.30),
        "q50": np_quantile_safe(scores, 0.50),
        "q55": np_quantile_safe(scores, 0.55),
        "q70": np_quantile_safe(scores, 0.70),
        "q85": np_quantile_safe(scores, 0.85),
        "avg": float(np.nanmean(scores)),
        "n": int(len(scores)),
    }
    if TARGET_COL in use.columns:
        out["bad_rate"] = float(use[TARGET_COL].mean())
    return out

STAGE_SCORE_REF = {stage: stage_score_reference(stage) for stage in ["A", "B", "C"]}

def operational_policy_config(stage: str) -> dict:
    cfg = dict(stage_config(stage))
    review_thr = float(first_existing(cfg, ["review_threshold_score_300_900"], float("nan")))
    reject_thr = float(first_existing(cfg, ["reject_threshold_score_300_900"], float("nan")))
    ref = STAGE_SCORE_REF.get(stage, {})

    out = {
        "policy_mode": first_existing(cfg, ["policy_mode"]),
        "recommended_role": first_existing(cfg, ["recommended_role"]),
        "review_threshold_score_300_900": review_thr,
        "reject_threshold_score_300_900": reject_thr,
        "small_offer_threshold_score_300_900": float("nan"),
        "policy_variant": "registry_default",
    }

    if stage == "A" and len(ref) > 0:
        # Make Stage A operationally realistic:
        # - route only the riskiest thin-file cases to review_or_decline
        # - keep the majority in starter-offer lanes
        out["review_threshold_score_300_900"] = float(ref.get("q85", review_thr))
        out["reject_threshold_score_300_900"] = float(ref.get("q85", reject_thr))
        out["small_offer_threshold_score_300_900"] = float(ref.get("q55", ref.get("q50", 700.0)))
        out["policy_variant"] = "stage_a_operational_override_v2"
        return out

    if stage == "B" and len(ref) > 0:
        out["small_offer_threshold_score_300_900"] = float(ref.get("q50", 700.0))
        return out

    return out

def detect_stage_from_row(row: dict) -> str:
    for key in ["stage", "segment_stage", "customer_stage"]:
        val = row.get(key)
        if isinstance(val, str) and val.strip().upper() in {"A", "B", "C"}:
            return val.strip().upper()

    dpd = first_existing(row, ["current_dpd", "max_dpd", "dpd_max", "days_past_due", "dpd"])
    if dpd is not None:
        try:
            if float(dpd) > 0:
                return "C"
        except Exception:
            pass

    has_delinquency = first_existing(row, ["has_dpd_positive", "has_any_dpd", "ever_dpd_positive"])
    if has_delinquency in [1, True, "1", "true", "True"]:
        return "C"

    tenure = first_existing(row, [
        "tenure_months", "months_since_first_loan", "months_on_book",
        "customer_tenure_months", "tenure_mob", "mob"
    ])
    if tenure is not None:
        try:
            tenure = float(tenure)
            if tenure <= 0:
                return "A"
            elif tenure <= 12:
                return "B"
            else:
                return "C"
        except Exception:
            pass

    return "A"

def get_stage_model_entry(stage: str) -> dict:
    return inference_pack.get("stage_models", {}).get(stage, {})

def extract_feature_list(model_entry: dict) -> list[str]:
    for key in ["kept_features", "feature_names", "features", "selected_features"]:
        val = model_entry.get(key)
        if isinstance(val, list) and len(val) > 0:
            return [str(x) for x in val]
    model = model_entry.get("model")
    if model is not None and hasattr(model, "feature_names_in_"):
        return [str(x) for x in list(model.feature_names_in_)]
    return []

def heuristic_score_from_payload(row: dict, stage: str) -> tuple[float, str]:
    ref = STAGE_SCORE_REF.get(stage, {})
    base = float(ref.get("q50", ref.get("avg", 600.0)))

    income = first_existing(row, ["main_income", "income", "monthly_income", "salary", "amt_income_total"])
    dpd = first_existing(row, ["current_dpd", "max_dpd", "days_past_due", "dpd"])
    tenure = first_existing(row, ["tenure_months", "months_since_first_loan", "months_on_book", "mob"])

    score = base
    notes = []

    try:
        if income is not None:
            income = float(income)
            if income < 5_000_000:
                score += 20
                notes.append("low_income")
            elif income >= 12_000_000:
                score -= 12
                notes.append("higher_income")
            elif income >= 8_000_000:
                score -= 5
                notes.append("mid_high_income")
    except Exception:
        pass

    try:
        if tenure is not None:
            tenure = float(tenure)
            if stage == "A" and tenure <= 0:
                notes.append("thin_file")
            elif stage == "B" and tenure <= 6:
                score += 5
                notes.append("early_behavior")
            elif stage == "C" and tenure >= 24:
                score += 4
                notes.append("mature_portfolio")
    except Exception:
        pass

    try:
        if dpd is not None:
            dpd = float(dpd)
            if dpd > 0:
                score += 35 if stage == "C" else 20
                notes.append("positive_dpd")
            if dpd >= 30:
                score += 20
                notes.append("severe_dpd")
    except Exception:
        pass

    score = float(min(900.0, max(300.0, score)))
    note = "heuristic_stage_distribution_proxy"
    if len(notes) > 0:
        note += ":" + ",".join(notes[:3])
    return score, note

def score_stage_row(row: dict, stage: str) -> dict:
    model_entry = get_stage_model_entry(stage)
    model = model_entry.get("model")
    feature_list = extract_feature_list(model_entry)

    x_df = pd.DataFrame([{f: row.get(f, np.nan) for f in feature_list}]) if len(feature_list) > 0 else pd.DataFrame([row])

    score_proba = float("nan")
    score_raw = float("nan")
    scoring_status = "unscored"
    scoring_note = None

    if model is not None:
        if hasattr(model, "predict_proba"):
            try:
                score_proba = float(model.predict_proba(x_df)[:, 1][0])
                scoring_status = "model_predict_proba"
            except Exception:
                pass

        if math.isnan(score_proba) and hasattr(model, "decision_function"):
            try:
                raw = float(model.decision_function(x_df)[0])
                score_raw = raw
                score_proba = float(sigmoid(raw))
                scoring_status = "model_decision_function"
            except Exception:
                pass

        if math.isnan(score_proba) and hasattr(model, "predict"):
            try:
                pred = float(model.predict(x_df)[0])
                score_proba = 0.90 if pred >= 1 else 0.10
                scoring_status = "model_class_only_fallback"
            except Exception:
                pass

    if math.isnan(score_proba):
        score_proba = first_existing(row, [PROBA_COL, "pred_proba", "prob_default", "pd_hat"], float("nan"))
        if not math.isnan(score_proba):
            scoring_status = "pre_scored_input"

    score_300_900 = proba_to_score_300_900(score_proba)
    if math.isnan(score_300_900):
        score_300_900 = first_existing(row, [SCORE_COL, "risk_score", "score"], float("nan"))
        if not math.isnan(score_300_900):
            score_proba = score_to_proba_300_900(score_300_900)
            scoring_status = "pre_scored_input"

    if math.isnan(score_300_900):
        score_300_900, scoring_note = heuristic_score_from_payload(row, stage)
        score_proba = score_to_proba_300_900(score_300_900)
        score_raw = score_proba
        scoring_status = "heuristic_stage_distribution_proxy"

    score_proba = clamp_proba(score_proba)
    if math.isnan(score_raw):
        score_raw = score_proba

    return {
        "stage": stage,
        "score_proba": score_proba,
        "score_raw": score_raw,
        "score_300_900": score_300_900,
        "feature_list_used": feature_list,
        "scoring_status": scoring_status,
        "scoring_note": scoring_note,
    }

def map_action_from_score(stage: str, score_300_900: float) -> dict:
    cfg = operational_policy_config(stage)
    review_thr = float(first_existing(cfg, ["review_threshold_score_300_900"], float("nan")))
    reject_thr = float(first_existing(cfg, ["reject_threshold_score_300_900"], float("nan")))
    small_offer_thr = float(first_existing(cfg, ["small_offer_threshold_score_300_900"], float("nan")))
    role = first_existing(cfg, ["recommended_role"], None)
    policy_mode = first_existing(cfg, ["policy_mode"], None)
    policy_variant = first_existing(cfg, ["policy_variant"], "registry_default")

    if stage == "A":
        if score_300_900 >= review_thr:
            zone = "review_or_decline"
            action = "review_or_decline"
            recommended_limit = 0
            recommended_tenor_months = None
            review_flag = True
        elif score_300_900 >= small_offer_thr:
            zone = "starter_loan_small"
            action = "starter_loan_small"
            recommended_limit = 5000000
            recommended_tenor_months = 3
            review_flag = False
        else:
            zone = "starter_loan_standard"
            action = "starter_loan_standard"
            recommended_limit = 10000000
            recommended_tenor_months = 6
            review_flag = False

    elif stage == "B":
        if score_300_900 >= reject_thr:
            zone = "reject_or_intensive_review"
            action = "reject_or_intensive_review"
            recommended_limit = 0
            recommended_tenor_months = None
            review_flag = True
        elif score_300_900 >= review_thr:
            zone = "manual_review"
            action = "manual_review"
            recommended_limit = 0
            recommended_tenor_months = None
            review_flag = True
        else:
            if math.isnan(small_offer_thr):
                ref = STAGE_SCORE_REF.get(stage, {})
                small_offer_thr = float(ref.get("q50", (300.0 + float(review_thr)) / 2.0 if not math.isnan(float(review_thr)) else 560.0))
            if score_300_900 <= small_offer_thr:
                zone = "approve_preferred"
                action = "approve_preferred"
                recommended_limit = 20000000
                recommended_tenor_months = 12
                review_flag = False
            else:
                zone = "approve_standard"
                action = "approve_standard"
                recommended_limit = 12000000
                recommended_tenor_months = 9
                review_flag = False

    else:
        if score_300_900 >= reject_thr:
            zone = "intensive_collection_priority"
            action = "intensive_collection_priority"
            recommended_limit = None
            recommended_tenor_months = None
            review_flag = True
        elif score_300_900 >= review_thr:
            zone = "high_risk_review_priority"
            action = "high_risk_review_priority"
            recommended_limit = None
            recommended_tenor_months = None
            review_flag = True
        else:
            zone = "monitor_or_standard_queue"
            action = "monitor_or_standard_queue"
            recommended_limit = None
            recommended_tenor_months = None
            review_flag = False

    return {
        "stage": stage,
        "policy_mode": policy_mode,
        "recommended_role": role,
        "policy_variant": policy_variant,
        "decision_zone": zone,
        "policy_action": action,
        "review_flag": review_flag,
        "recommended_limit": recommended_limit,
        "recommended_tenor_months": recommended_tenor_months,
        "review_threshold_score_300_900": review_thr,
        "reject_threshold_score_300_900": reject_thr,
    }

def derive_reason_codes(row: dict, stage: str, feature_list: list[str]) -> list[str]:
    reasons = []

    income = first_existing(row, ["main_income", "income", "monthly_income", "salary", "amt_income_total"])
    if income is not None:
        try:
            if float(income) < 5000000:
                reasons.append("low_income_signal")
            elif float(income) >= 12000000:
                reasons.append("higher_income_signal")
        except Exception:
            pass

    dpd = first_existing(row, ["current_dpd", "max_dpd", "days_past_due", "dpd"])
    if dpd is not None:
        try:
            if float(dpd) > 0:
                reasons.append("positive_dpd_signal")
        except Exception:
            pass

    tenure = first_existing(row, ["tenure_months", "months_since_first_loan", "months_on_book"])
    if tenure is not None:
        try:
            if float(tenure) <= 0:
                reasons.append("new_to_bank_thin_file")
            elif float(tenure) <= 12:
                reasons.append("early_behavior_history")
            else:
                reasons.append("mature_customer_history")
        except Exception:
            pass

    for f in feature_list[:10]:
        val = row.get(f, None)
        if val is not None and not (isinstance(val, float) and math.isnan(val)):
            reasons.append(f"feature_present:{f}")

    out = []
    for r in reasons:
        if r not in out:
            out.append(r)
    return out[:3]

def infer_single_payload(payload: dict) -> dict:
    stage = detect_stage_from_row(payload)
    scored = score_stage_row(payload, stage)
    action = map_action_from_score(stage, scored["score_300_900"])
    reason_codes = derive_reason_codes(payload, stage, scored.get("feature_list_used", []))

    return {
        "customer_id": payload.get("customer_id"),
        "stage": stage,
        "score_proba": scored["score_proba"],
        "score_raw": scored["score_raw"],
        "score_300_900": scored["score_300_900"],
        "scoring_status": scored["scoring_status"],
        "scoring_note": scored["scoring_note"],
        "policy_mode": action["policy_mode"],
        "policy_variant": action["policy_variant"],
        "recommended_role": action["recommended_role"],
        "decision_zone": action["decision_zone"],
        "policy_action": action["policy_action"],
        "recommended_limit": action["recommended_limit"],
        "recommended_tenor_months": action["recommended_tenor_months"],
        "review_flag": action["review_flag"],
        "reason_code_1": reason_codes[0] if len(reason_codes) > 0 else None,
        "reason_code_2": reason_codes[1] if len(reason_codes) > 1 else None,
        "reason_code_3": reason_codes[2] if len(reason_codes) > 2 else None,
    }

def infer_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = [infer_single_payload(rec) for rec in df.to_dict(orient="records")]
    return pd.DataFrame(rows)

def apply_zone_from_thresholds(df: pl.DataFrame, stage: str, score_col: str = SCORE_COL) -> pl.DataFrame:
    cfg = operational_policy_config(stage)
    review_thr = float(first_existing(cfg, ["review_threshold_score_300_900"], float("nan")))
    reject_thr = float(first_existing(cfg, ["reject_threshold_score_300_900"], float("nan")))
    small_offer_thr = float(first_existing(cfg, ["small_offer_threshold_score_300_900"], float("nan")))

    if stage == "A":
        if math.isnan(small_offer_thr):
            ref = STAGE_SCORE_REF.get(stage, {})
            small_offer_thr = float(ref.get("q55", ref.get("q50", 700.0)))
        return (
            df.with_columns(
                pl.when(pl.col(score_col) >= review_thr)
                .then(pl.lit("review_or_decline"))
                .when(pl.col(score_col) >= small_offer_thr)
                .then(pl.lit("starter_loan_small"))
                .otherwise(pl.lit("starter_loan_standard"))
                .alias("decision_zone")
            )
        )

    if stage == "B":
        if math.isnan(small_offer_thr):
            ref = STAGE_SCORE_REF.get(stage, {})
            small_offer_thr = float(ref.get("q50", (300.0 + review_thr) / 2.0 if not math.isnan(review_thr) else 560.0))
        return (
            df.with_columns(
                pl.when(pl.col(score_col) >= reject_thr)
                .then(pl.lit("reject_or_intensive_review"))
                .when(pl.col(score_col) >= review_thr)
                .then(pl.lit("manual_review"))
                .when(pl.col(score_col) <= small_offer_thr)
                .then(pl.lit("approve_preferred"))
                .otherwise(pl.lit("approve_standard"))
                .alias("decision_zone")
            )
        )

    return (
        df.with_columns(
            pl.when(pl.col(score_col) >= reject_thr)
            .then(pl.lit("intensive_collection_priority"))
            .when(pl.col(score_col) >= review_thr)
            .then(pl.lit("high_risk_review_priority"))
            .otherwise(pl.lit("monitor_or_standard_queue"))
            .alias("decision_zone")
        )
    )

def zone_kpi_summary(df: pl.DataFrame, stage: str, score_col: str = SCORE_COL) -> pl.DataFrame:
    cols = [c for c in [TARGET_COL, score_col] if c in df.columns]
    if score_col not in cols:
        return pl.DataFrame()
    use = df.select(cols).drop_nulls()
    if use.height == 0:
        return pl.DataFrame()

    zoned = apply_zone_from_thresholds(use, stage=stage, score_col=score_col)
    out = (
        zoned.group_by("decision_zone")
        .agg([
            pl.len().alias("n_cases"),
            pl.mean(score_col).alias("avg_score_300_900"),
            pl.mean(TARGET_COL).alias("bad_rate") if TARGET_COL in use.columns else pl.lit(None).alias("bad_rate"),
        ])
        .with_columns((pl.col("n_cases") / pl.col("n_cases").sum()).alias("case_share"))
        .sort("avg_score_300_900", descending=True)
        .with_columns(pl.lit(stage).alias("stage"))
        .select(["stage", "decision_zone", "n_cases", "case_share", "avg_score_300_900", "bad_rate"])
    )
    return out

def build_streamlit_input_schema() -> pl.DataFrame:
    rows = [
        {"stage": "ALL", "field_name": "customer_id", "field_order": 1, "data_type_hint": "string", "required_for_model": False, "ui_group": "identity"},
        {"stage": "ALL", "field_name": "tenure_months", "field_order": 2, "data_type_hint": "numeric", "required_for_model": False, "ui_group": "stage_router"},
        {"stage": "ALL", "field_name": "max_dpd", "field_order": 3, "data_type_hint": "numeric", "required_for_model": False, "ui_group": "stage_router"},
        {"stage": "ALL", "field_name": "main_income", "field_order": 4, "data_type_hint": "numeric", "required_for_model": False, "ui_group": "business_context"},
    ]
    seen = {(r["stage"], r["field_name"]) for r in rows}
    for stage in ["A", "B", "C"]:
        feature_list = extract_feature_list(get_stage_model_entry(stage))
        for idx, feat in enumerate(feature_list, start=10):
            key = (stage, feat)
            if key in seen:
                continue
            rows.append({
                "stage": stage,
                "field_name": feat,
                "field_order": idx,
                "data_type_hint": "numeric_or_nullable",
                "required_for_model": True,
                "ui_group": "model_inputs",
            })
            seen.add(key)
    return pl.DataFrame(rows).sort(["stage", "field_order"])


In [5]:

# =========================================================
# 4. Product pack overview
# =========================================================
product_pack_overview = pl.DataFrame([
    {
        "champion_decision_stage": model_registry_manifest.get("champion_decision_stage"),
        "registry_name": model_registry_manifest.get("registry_name"),
        "source_notebook_4_dir": model_registry_manifest.get("source_notebook_4_dir"),
        "source_notebook_5_dir": str(INPUT_DIR_NB5),
        "global_note": model_registry_manifest.get("global_note"),
    }
])

stage_role_overview = (
    registry_df
    .select([
        "stage",
        "registry_role",
        "policy_mode",
        "recommended_role",
        "ready_for_nb5_business_strict",
        "ready_for_product_scoring_api",
        "auc_valid",
        "ks_valid",
        "review_threshold_score_300_900",
        "reject_threshold_score_300_900",
    ])
    .sort("stage")
)

display(product_pack_overview)
display(stage_role_overview)


champion_decision_stage,registry_name,source_notebook_4_dir,source_notebook_5_dir,global_note
str,str,str,str,str
"""B""","""homecredit_proxy_stage_registr…","""/kaggle/input/notebooks/hongvi…","""/kaggle/input/notebooks/hongvi…","""Notebook 5 benchmark + registr…"


stage,registry_role,policy_mode,recommended_role,ready_for_nb5_business_strict,ready_for_product_scoring_api,auc_valid,ks_valid,review_threshold_score_300_900,reject_threshold_score_300_900
str,str,str,str,bool,bool,f64,f64,f64,f64
"""A""","""active_guarded_model""","""starter_offer_only""","""starter_offer""",false,true,0.546369,0.122847,814.777871,829.939654
"""B""","""champion_decision_model""","""full_decision""","""decision_champion""",true,true,0.627878,0.22536,742.761886,746.833473
"""C""","""active_guarded_model""","""guardrail_only""","""collection_guardrail""",false,true,0.627189,0.23954,724.997459,796.199662


In [6]:

# =========================================================
# 5. Unified inference demo on sample payloads
# =========================================================
sample_payloads = [
    {
        "customer_id": "demo_A_001",
        "tenure_months": 0,
        "main_income": 4500000,
        "max_dpd": 0,
    },
    {
        "customer_id": "demo_B_001",
        "tenure_months": 8,
        "main_income": 12000000,
        "max_dpd": 0,
    },
    {
        "customer_id": "demo_C_001",
        "tenure_months": 26,
        "main_income": 9000000,
        "max_dpd": 15,
    },
]

sample_request_df = pd.DataFrame(sample_payloads)
sample_response_df = infer_dataframe(sample_request_df)

display(sample_request_df)
display(sample_response_df)


,customer_id,tenure_months,main_income,max_dpd
0,demo_A_001,0,4500000,0
1,demo_B_001,8,12000000,0
2,demo_C_001,26,9000000,15


,customer_id,stage,score_proba,score_raw,score_300_900,scoring_status,scoring_note,policy_mode,policy_variant,recommended_role,decision_zone,policy_action,recommended_limit,recommended_tenor_months,review_flag,reason_code_1,reason_code_2,reason_code_3
0,demo_A_001,A,0.891296,0.891296,834.777871,heuristic_stage_distribution_proxy,"heuristic_stage_distribution_proxy:low_income,...",starter_offer_only,stage_a_operational_override_v2,starter_offer,review_or_decline,review_or_decline,0.0,NaN,True,low_income_signal,new_to_bank_thin_file,None
1,demo_B_001,B,0.704990,0.704990,722.994077,heuristic_stage_distribution_proxy,heuristic_stage_distribution_proxy:higher_income,full_decision,registry_default,decision_champion,approve_preferred,approve_preferred,20000000.0,12.0,False,higher_income_signal,early_behavior_history,None
2,demo_C_001,C,0.753443,0.753443,752.065510,heuristic_stage_distribution_proxy,heuristic_stage_distribution_proxy:mid_high_in...,guardrail_only,registry_default,collection_guardrail,high_risk_review_priority,high_risk_review_priority,NaN,NaN,True,positive_dpd_signal,mature_customer_history,None


In [7]:
# =========================================================
# 6. Policy simulation on scored validation sets
# =========================================================
zone_sim_tables = []
stage_policy_summary_rows = []

for stage in ["A", "B", "C"]:
    df = stage_valid_scored.get(stage)
    if df is None or SCORE_COL not in df.columns:
        print(f"[WARN] Skip simulation for stage {stage}: missing scored validation table or {SCORE_COL}")
        continue

    sim_tbl = zone_kpi_summary(df, stage=stage, score_col=SCORE_COL)
    if sim_tbl.height == 0:
        print(f"[WARN] Skip simulation for stage {stage}: empty KPI summary")
        continue

    zone_sim_tables.append(sim_tbl)

    sim_dicts = sim_tbl.to_dicts()
    review_cases = 0
    approve_cases = 0
    reject_cases = 0

    for row in sim_dicts:
        zone = row["decision_zone"]
        n_cases = int(row["n_cases"])
        if zone in ["manual_review", "high_risk_review_priority", "decline_or_manual_review", "review_or_decline"]:
            review_cases += n_cases
        if zone in ["starter_loan_small", "starter_loan_standard", "approve_standard", "approve_preferred", "monitor_or_standard_queue"]:
            approve_cases += n_cases
        if zone in ["reject_or_intensive_review", "intensive_collection_priority"]:
            reject_cases += n_cases

    total_cases = int(sum(int(r["n_cases"]) for r in sim_dicts))
    total_bad_est = float(sum((r.get("bad_rate") or 0.0) * int(r["n_cases"]) for r in sim_dicts))
    weighted_bad_rate = safe_div(total_bad_est, total_cases)

    stage_policy_summary_rows.append({
        "stage": stage,
        "n_cases": total_cases,
        "review_workload_n": review_cases,
        "review_workload_rate": safe_div(review_cases, total_cases),
        "approve_or_continue_n": approve_cases,
        "approve_or_continue_rate": safe_div(approve_cases, total_cases),
        "reject_or_intensive_n": reject_cases,
        "reject_or_intensive_rate": safe_div(reject_cases, total_cases),
        "weighted_bad_rate_estimate": weighted_bad_rate,
    })

policy_zone_simulation_df = (
    pl.concat(zone_sim_tables, how="vertical")
    .sort(["stage", "avg_score_300_900"], descending=[False, True])
    if len(zone_sim_tables) > 0 else pl.DataFrame(schema={
        "stage": pl.String,
        "decision_zone": pl.String,
        "n_cases": pl.Int64,
        "case_share": pl.Float64,
        "avg_score_300_900": pl.Float64,
        "bad_rate": pl.Float64,
    })
)

policy_stage_summary_df = (
    pl.DataFrame(stage_policy_summary_rows).sort("stage")
    if len(stage_policy_summary_rows) > 0 else pl.DataFrame(schema={
        "stage": pl.String,
        "n_cases": pl.Int64,
        "review_workload_n": pl.Int64,
        "review_workload_rate": pl.Float64,
        "approve_or_continue_n": pl.Int64,
        "approve_or_continue_rate": pl.Float64,
        "reject_or_intensive_n": pl.Int64,
        "reject_or_intensive_rate": pl.Float64,
        "weighted_bad_rate_estimate": pl.Float64,
    })
)

display(policy_zone_simulation_df)
display(policy_stage_summary_df)


stage,decision_zone,n_cases,case_share,avg_score_300_900,bad_rate
str,str,u32,f64,f64,f32
"""A""","""review_or_decline""",9559,0.150009,837.254442,0.034209
"""A""","""starter_loan_small""",43407,0.681183,814.974868,0.017785
"""A""","""starter_loan_standard""",10757,0.168809,760.78498,0.02352
"""B""","""reject_or_intensive_review""",21701,0.100002,753.336979,0.036542
"""B""","""manual_review""",32551,0.15,744.549159,0.029861
"""B""","""approve_standard""",54251,0.249998,738.304054,0.030211
"""B""","""approve_preferred""",108503,0.5,728.879217,0.01258
"""C""","""intensive_collection_priority""",1231,0.050037,798.167993,0.120227
"""C""","""high_risk_review_priority""",3690,0.149988,743.2397,0.115718


stage,n_cases,review_workload_n,review_workload_rate,approve_or_continue_n,approve_or_continue_rate,reject_or_intensive_n,reject_or_intensive_rate,weighted_bad_rate_estimate
str,i64,i64,f64,i64,f64,i64,f64,f64
"""A""",63723,9559,0.150009,54164,0.849991,0,0.0,0.021217
"""B""",217006,32551,0.15,162754,0.749998,21701,0.100002,0.021976
"""C""",24602,3690,0.149988,19681,0.799976,1231,0.050037,0.073205


In [8]:

# =========================================================
# 7. Build product handoff pack
# =========================================================
streamlit_input_schema_df = build_streamlit_input_schema()

policy_rules = {
    "score_semantics": "Higher score_300_900 means higher risk.",
    "champion_decision_stage": model_registry_manifest.get("champion_decision_stage"),
    "policy_design_note": (
        "Stage A uses an operational override in notebook 6 to reduce review overload and keep "
        "the onboarding flow practical. Stages B and C stay close to registry thresholds."
    ),
    "stages": {},
}

for stage in ["A", "B", "C"]:
    cfg = operational_policy_config(stage)
    policy_rules["stages"][stage] = {
        "policy_mode": first_existing(cfg, ["policy_mode"]),
        "recommended_role": first_existing(cfg, ["recommended_role"]),
        "policy_variant": first_existing(cfg, ["policy_variant"]),
        "review_threshold_score_300_900": first_existing(cfg, ["review_threshold_score_300_900"]),
        "reject_threshold_score_300_900": first_existing(cfg, ["reject_threshold_score_300_900"]),
        "small_offer_threshold_score_300_900": first_existing(cfg, ["small_offer_threshold_score_300_900"]),
        "zone_labels": (
            ["review_or_decline", "starter_loan_small", "starter_loan_standard"] if stage == "A" else
            ["reject_or_intensive_review", "manual_review", "approve_standard", "approve_preferred"] if stage == "B" else
            ["intensive_collection_priority", "high_risk_review_priority", "monitor_or_standard_queue"]
        ),
    }

inference_service_pack = {
    "pack_name": "homecredit_proxy_unified_inference_pack_v2",
    "source_notebook_4_dir": str(INPUT_DIR_NB4),
    "source_notebook_5_dir": str(INPUT_DIR_NB5),
    "work_dir_notebook_6": str(WORK_DIR),
    "model_registry_manifest": model_registry_manifest,
    "product_scoring_manifest": product_scoring_manifest,
    "policy_rules": policy_rules,
    "registry_table": registry_df.to_dicts(),
    "policy_recommendation_table": policy_recommendation_df.to_dicts(),
    "streamlit_input_schema": streamlit_input_schema_df.to_dicts(),
    "stage_score_reference": STAGE_SCORE_REF,
    "inference_pack": inference_pack,
    "registry_pack": registry_pack,
}

business_glossary = """# Business Glossary

- **Stage A**: New-to-bank / thin-file onboarding segment. Output should be interpreted as starter-offer recommendation, not a hard approval model.
- **Stage B**: Primary decision stage with the strongest benchmark performance in the current system.
- **Stage C**: Mature / delinquent / monitoring segment. Use for collection prioritization or guardrail workflows.
- **score_proba**: Risk probability-like output from the model. It is not guaranteed to be calibrated PD.
- **score_300_900**: Linear transformation of score_proba into a product-facing score where higher means riskier.
- **decision_zone**: Business-friendly operating segment derived from stage-specific score thresholds.
- **policy_action**: Recommended operational action for the zone.
- **review_flag**: Whether the case should be routed to manual review or high-priority queue.
- **scoring_status**: Indicates whether the response came from a real model score, a pre-scored input, or a transparent heuristic proxy.
- **policy_variant**: Indicates whether the decision used the registry-default thresholds or the notebook-6 operational override.
"""

display(streamlit_input_schema_df.head(20))
print(json.dumps(policy_rules, indent=2, ensure_ascii=False))


stage,field_name,field_order,data_type_hint,required_for_model,ui_group
str,str,i64,str,bool,str
"""ALL""","""customer_id""",1,"""string""",false,"""identity"""
"""ALL""","""tenure_months""",2,"""numeric""",false,"""stage_router"""
"""ALL""","""max_dpd""",3,"""numeric""",false,"""stage_router"""
"""ALL""","""main_income""",4,"""numeric""",false,"""business_context"""


{
  "score_semantics": "Higher score_300_900 means higher risk.",
  "champion_decision_stage": "B",
  "policy_design_note": "Stage A uses an operational override in notebook 6 to reduce review overload and keep the onboarding flow practical. Stages B and C stay close to registry thresholds.",
  "stages": {
    "A": {
      "policy_mode": "starter_offer_only",
      "recommended_role": "starter_offer",
      "policy_variant": "stage_a_operational_override_v2",
      "review_threshold_score_300_900": 821.9200038933097,
      "reject_threshold_score_300_900": 821.9200038933097,
      "small_offer_threshold_score_300_900": 814.7778705540012,
      "zone_labels": [
        "review_or_decline",
        "starter_loan_small",
        "starter_loan_standard"
      ]
    },
    "B": {
      "policy_mode": "full_decision",
      "recommended_role": "decision_champion",
      "policy_variant": "registry_default",
      "review_threshold_score_300_900": 742.76188562518,
      "reject_threshold_scor

In [9]:

# =========================================================
# 8. Export notebook 6 artifacts
# =========================================================
saved_paths = []

zone_sim_path = WORK_DIR / "policy_zone_simulation.csv"
policy_zone_simulation_df.write_csv(zone_sim_path)
saved_paths.append(str(zone_sim_path))

stage_summary_path = WORK_DIR / "policy_stage_summary.csv"
policy_stage_summary_df.write_csv(stage_summary_path)
saved_paths.append(str(stage_summary_path))

streamlit_schema_path = WORK_DIR / "streamlit_input_schema.csv"
streamlit_input_schema_df.write_csv(streamlit_schema_path)
saved_paths.append(str(streamlit_schema_path))

policy_rules_path = WORK_DIR / "policy_rules.json"
with open(policy_rules_path, "w", encoding="utf-8") as f:
    json.dump(policy_rules, f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(policy_rules_path))

sample_request_path = WORK_DIR / "sample_request_payload.json"
with open(sample_request_path, "w", encoding="utf-8") as f:
    json.dump(sample_payloads, f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(sample_request_path))

sample_response_path = WORK_DIR / "sample_response_payload.json"
with open(sample_response_path, "w", encoding="utf-8") as f:
    json.dump(sample_response_df.to_dict(orient="records"), f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(sample_response_path))

glossary_path = WORK_DIR / "business_glossary.md"
with open(glossary_path, "w", encoding="utf-8") as f:
    f.write(business_glossary)
saved_paths.append(str(glossary_path))

service_pack_path = WORK_DIR / "unified_inference_service.joblib"
joblib.dump(inference_service_pack, service_pack_path)
saved_paths.append(str(service_pack_path))

manifest_path = WORK_DIR / "product_pack_manifest.json"
product_pack_manifest = {
    "pack_name": "homecredit_proxy_product_pack_v1",
    "source_notebook_4_dir": str(INPUT_DIR_NB4),
    "source_notebook_5_dir": str(INPUT_DIR_NB5),
    "work_dir_notebook_6": str(WORK_DIR),
    "exports": saved_paths,
    "score_note": "Higher score_300_900 means higher risk.",
    "deployment_note": (
        "Load unified_inference_service.joblib in Streamlit/API. "
        "Use policy_rules.json for action mapping and streamlit_input_schema.csv for UI fields."
    ),
}
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(product_pack_manifest, f, ensure_ascii=False, indent=2, default=str)
saved_paths.append(str(manifest_path))

print("Saved notebook 6 artifacts:")
for p in saved_paths:
    print("-", p)


Saved notebook 6 artifacts:
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/policy_zone_simulation.csv
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/policy_stage_summary.csv
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/streamlit_input_schema.csv
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/policy_rules.json
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/sample_request_payload.json
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/sample_response_payload.json
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/business_glossary.md
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/unified_inference_service.joblib
- /kaggle/working/homecredit_proxy_notebook_06_product_pack/product_pack_manifest.json


In [10]:

# =========================================================
# 9. Final summary for handoff
# =========================================================
final_summary_df = (
    registry_df
    .select([
        "stage",
        "registry_role",
        "policy_mode",
        "recommended_role",
        "auc_valid",
        "ks_valid",
        "review_threshold_score_300_900",
        "reject_threshold_score_300_900",
    ])
    .join(
        policy_stage_summary_df.select([
            "stage",
            "review_workload_rate",
            "approve_or_continue_rate",
            "reject_or_intensive_rate",
            "weighted_bad_rate_estimate",
        ]),
        on="stage",
        how="left",
    )
    .sort("stage")
)

display(final_summary_df)
print("Notebook 6 complete. Product pack and policy simulator artifacts are ready.")


stage,registry_role,policy_mode,recommended_role,auc_valid,ks_valid,review_threshold_score_300_900,reject_threshold_score_300_900,review_workload_rate,approve_or_continue_rate,reject_or_intensive_rate,weighted_bad_rate_estimate
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""A""","""active_guarded_model""","""starter_offer_only""","""starter_offer""",0.546369,0.122847,814.777871,829.939654,0.150009,0.849991,0.0,0.021217
"""B""","""champion_decision_model""","""full_decision""","""decision_champion""",0.627878,0.22536,742.761886,746.833473,0.15,0.749998,0.100002,0.021976
"""C""","""active_guarded_model""","""guardrail_only""","""collection_guardrail""",0.627189,0.23954,724.997459,796.199662,0.149988,0.799976,0.050037,0.073205


Notebook 6 complete. Product pack and policy simulator artifacts are ready.
